In [ ]:
!nvidia-smi

Mon Dec  8 15:20:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [ ]:
import cupy as cp
print(cp.cuda.runtime.getDeviceCount(), "GPU(s) detected")

1 GPU(s) detected


In [ ]:
import os

print("Exists:", os.path.exists("TMDB_movie_dataset_v112.csv"))

Exists: False


In [ ]:
import cupy as cp
cp.show_config()

OS                           : Linux-6.6.105+-x86_64-with-glibc2.35
Python Version               : 3.12.12
CuPy Version                 : 13.6.0
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 2.0.2
SciPy Version                : 1.16.3
Cython Build Version         : 3.0.12
Cython Runtime Version       : 3.0.12
CUDA Root                    : /usr/local/cuda
nvcc PATH                    : /usr/local/cuda/bin/nvcc
CUDA Build Version           : 12090
CUDA Driver Version          : 12040
CUDA Runtime Version         : 12090 (linked to CuPy) / 12060 (locally installed)
CUDA Extra Include Dirs      : ['/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/include']
cuBLAS Version               : (available)
cuFFT Version                : 11300
cuRAND Version               : 10307
cuSOLVER Version             : (11, 6, 3)
cuSPARSE Version             : (available)
NVRTC Version                : (12, 6)
Thrust Version               : 200800
CUB Build Version   

In [ ]:
!pip install numba cuda-python -q

In [ ]:
!pip install cupy-cuda12x numba pandas

In [ ]:
!pip install cupy-cuda12x

In [ ]:
!apt-get update -qq
!apt-get install -y openmpi-bin libopenmpi-dev
!pip install mpi4py cupy-cuda12x

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.


In [ ]:
!pip install mpi4py cupy-cuda12x

In [ ]:
    !apt-get update
    !apt-get install -y openmpi-bin libopenmpi-dev
    !pip install mpi4py

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,513 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/p

#MPI blocking
##without Tuning

In [ ]:
%%writefile mpi_gpu_tmdb.py
from mpi4py import MPI
import cupy as cp
import numpy as np
import csv
import sys
import time

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

def trim_tail_only(s: str) -> str:
    return s.strip()

def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp

def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0

def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0

def get_column_indices(header_row):
    idxTitle = idxRating = idxRelease = idxRev = -1
    idxBudget = idxPop = idxGenre = -1
    for i, name in enumerate(header_row):
        col = trim_tail_only(name).lower()
        if col == "title":
            idxTitle = i
        elif col == "vote_average":
            idxRating = i
        elif col == "revenue":
            idxRev = i
        elif col == "budget":
            idxBudget = i
        elif col == "popularity":
            idxPop = i
        elif col == "release_date":
            idxRelease = i
        elif col == "genres":
            idxGenre = i
    return idxTitle, idxRating, idxRelease, idxRev, idxBudget, idxPop, idxGenre

def count_total_rows(dataset_path):
    total = 0
    with open(dataset_path, "r", encoding="latin-1") as f:
        reader = csv.reader(f)
        try:
            _ = next(reader)
        except StopIteration:
            return 0
        for _ in reader:
            total += 1
    return total

def mpi_gpu_movie_analysis(dataset_path, partition_mode="row", block_size_rows=1024):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if rank == 0:
        print(">>> [DEBUG] mpi_gpu_movie_analysis started", flush=True)
        print(f">>> [DEBUG] dataset_path = {dataset_path}", flush=True)
        print(f">>> [DEBUG] partition_mode = {partition_mode}, block_size_rows = {block_size_rows}", flush=True)

    total_start = MPI.Wtime()
    comm_time_local = 0.0
    comp_time_local = 0.0
    gpu_compute_only = 0.0

    if rank == 0:
        total_rows = count_total_rows(dataset_path)
        print(f">>> [DEBUG] Rank 0 counted total_rows = {total_rows}", flush=True)
    else:
        total_rows = None

    t0 = MPI.Wtime()
    total_rows = comm.bcast(total_rows, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    if total_rows == 0:
        if rank == 0:
            print("Empty file.", flush=True)
        return

    if partition_mode == "row":
        base = total_rows // size
        rem  = total_rows % size
        start_row = rank * base + min(rank, rem)
        end_row   = start_row + base + (1 if rank < rem else 0)
    elif partition_mode == "block-cyclic":
        start_row = 0
        end_row = total_rows
    else:
        raise ValueError("partition_mode must be 'row' or 'block-cyclic'")

    print(f">>> [DEBUG] Rank {rank} will process rows in [{start_row}, {end_row}) with mode={partition_mode}",
          flush=True)

    if rank == 0:
        print(f"[MPI GPU] Using dataset: {dataset_path}", flush=True)
        print(f"[MPI GPU] Total rows (movies) = {total_rows}", flush=True)
        print(f"[MPI GPU] Partition mode = {partition_mode}, block_size_rows = {block_size_rows}", flush=True)
        print(f"[MPI GPU] World size (processes) = {size}", flush=True)
    comm.Barrier()

    titles_local = []
    years_local = []
    ratings_local = []
    revenues_local = []
    pops_local = []
    global_index_local = []

    genre_to_id_local = {}
    id_to_genre_local = []
    genre_ids_local = []
    genre_ratings_local = []

    comp_start_cpu = MPI.Wtime()

    with open(dataset_path, "r", encoding="latin-1") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            header = []

        idxTitle, idxRating, idxRelease, idxRev, idxBudget, idxPop, idxGenre = get_column_indices(header)

        global_row_idx = 0

        for row in reader:
            if len(row) < len(header):
                continue

            if partition_mode == "row":
                if not (start_row <= global_row_idx < end_row):
                    global_row_idx += 1
                    continue
            else:
                block_id = global_row_idx // block_size_rows
                owner = block_id % size
                if owner != rank:
                    global_row_idx += 1
                    continue

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles_local.append(title)
            years_local.append(year)
            ratings_local.append(rating)
            revenues_local.append(rev)
            pops_local.append(pop)
            global_index_local.append(global_row_idx)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id_local and len(genre_to_id_local) < MAX_GENRES:
                        gid = len(id_to_genre_local)
                        genre_to_id_local[first] = gid
                        id_to_genre_local.append(first)
                    gid = genre_to_id_local.get(first, None)
                    if gid is not None:
                        genre_ids_local.append(gid)
                        genre_ratings_local.append(rating)

            global_row_idx += 1

    comp_end_cpu = MPI.Wtime()

    print(f">>> [DEBUG] Rank {rank} local rows read = {len(titles_local)}", flush=True)

    years_np = np.array(years_local, dtype=np.int32)
    ratings_np = np.array(ratings_local, dtype=np.float64)
    revenues_np = np.array(revenues_local, dtype=np.float64)
    pops_np = np.array(pops_local, dtype=np.float64)

    if len(id_to_genre_local) > 0:
        genre_ids_np = np.array(genre_ids_local, dtype=np.int32)
        genre_ratings_np = np.array(genre_ratings_local, dtype=np.float64)
    else:
        genre_ids_np = None
        genre_ratings_np = None

    if years_np.size > 0:
        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if genre_ids_np is not None:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(year_idx, weights=rating_valid, minlength=YEAR_RANGE).astype(cp.float64)
        d_count_year = cp.bincount(year_idx, minlength=YEAR_RANGE).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=len(id_to_genre_local)
            ).astype(cp.float64)
            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=len(id_to_genre_local)
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_end = time.perf_counter()
        gpu_compute_only = gpu_end - gpu_start

        sum_year_local = cp.asnumpy(d_sum_year)
        count_year_local = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_local = cp.asnumpy(d_sum_genre)
            count_genre_local = cp.asnumpy(d_count_genre)
        else:
            sum_genre_local = np.zeros(0, dtype=np.float64)
            count_genre_local = np.zeros(0, dtype=np.int64)

        local_top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        local_top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)
    else:
        sum_year_local = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_local = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_local = np.zeros(0, dtype=np.float64)
        count_genre_local = np.zeros(0, dtype=np.int64)
        local_top_rev_idx = np.array([], dtype=int)
        local_top_pop_idx = np.array([], dtype=int)

    comp_time_local = (comp_end_cpu - comp_start_cpu) + gpu_compute_only

    local_top_rev = []
    for idx in local_top_rev_idx:
        if 0 <= idx < len(titles_local):
            local_top_rev.append((
                revenues_local[idx],
                titles_local[idx],
                global_index_local[idx]
            ))

    local_top_pop = []
    for idx in local_top_pop_idx:
        if 0 <= idx < len(titles_local):
            local_top_pop.append((
                pops_local[idx],
                titles_local[idx],
                years_local[idx],
                global_index_local[idx]
            ))

    sum_year_global = np.zeros_like(sum_year_local)
    count_year_global = np.zeros_like(count_year_local)

    t0 = MPI.Wtime()
    comm.Reduce(sum_year_local, sum_year_global, op=MPI.SUM, root=0)
    comm.Reduce(count_year_local, count_year_global, op=MPI.SUM, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    t0 = MPI.Wtime()
    all_top_rev = comm.gather(local_top_rev, root=0)
    all_top_pop = comm.gather(local_top_pop, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    local_genre_payload = (id_to_genre_local, sum_genre_local, count_genre_local)
    t0 = MPI.Wtime()
    all_genre_payloads = comm.gather(local_genre_payload, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    total_end = MPI.Wtime()
    total_time_local = total_end - total_start

    comm_time_global = comm.reduce(comm_time_local, op=MPI.MAX, root=0)
    comp_time_global = comm.reduce(comp_time_local, op=MPI.MAX, root=0)
    gpu_time_global  = comm.reduce(gpu_compute_only, op=MPI.MAX, root=0)
    total_time_global = comm.reduce(total_time_local, op=MPI.MAX, root=0)

    if rank == 0:
        flat_top_rev = []
        for lst in all_top_rev:
            flat_top_rev.extend(lst)
        flat_top_rev.sort(key=lambda x: x[0], reverse=True)
        flat_top_rev = flat_top_rev[:MAX_TOP]

        flat_top_pop = []
        for lst in all_top_pop:
            flat_top_pop.extend(lst)
        flat_top_pop.sort(key=lambda x: x[0], reverse=True)
        flat_top_pop = flat_top_pop[:MAX_TOP]

        global_genre_map = {}
        global_sum_genre = []
        global


Overwriting mpi_gpu_tmdb.py


In [ ]:
!python3 -u mpi_gpu_tmdb.py /content/TMDB_movie_dataset_v112.csv row


>>> [DEBUG] __main__ reached
>>> [DEBUG] sys.argv = ['mpi_gpu_tmdb.py', '/content/TMDB_movie_dataset_v112.csv', 'row']
>>> [DEBUG] mpi_gpu_movie_analysis started
>>> [DEBUG] dataset_path = /content/TMDB_movie_dataset_v112.csv
>>> [DEBUG] partition_mode = row, block_size_rows = 1024
>>> [DEBUG] Rank 0 counted total_rows = 1048575
>>> [DEBUG] Rank 0 will process rows in [0, 1048575) with mode=row
[MPI GPU] Using dataset: /content/TMDB_movie_dataset_v112.csv
[MPI GPU] Total rows (movies) = 1048575
[MPI GPU] Partition mode = row, block_size_rows = 1024
[MPI GPU] World size (processes) = 1
>>> [DEBUG] Rank 0 local rows read = 1048575
GPU Version – TMDB Movie Analysis (MPI + GPU)
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 â*: .ï½¡. o(â§â½â¦)o .ï½¡.:*â UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie"

In [ ]:
!ls


hello_mpi.py	 mpi_hello.py  TMDB_movie_dataset_v112.csv
mpi_gpu_tmdb.py  sample_data


In [ ]:
!ls /content/TMDB_movie_dataset_v112.csv


/content/TMDB_movie_dataset_v112.csv


#MPI non Blocking
##without tunning

In [ ]:
%%writefile mpinon_gpu_tmdb.py
from mpi4py import MPI
import cupy as cp
import numpy as np
import csv
import sys
import time

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

def trim_tail_only(s: str) -> str:
    return s.strip()

def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp

def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0

def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0

def get_column_indices(header_row):
    idxTitle = idxRating = idxRelease = idxRev = -1
    idxBudget = idxPop = idxGenre = -1
    for i, name in enumerate(header_row):
        col = trim_tail_only(name).lower()
        if col == "title":
            idxTitle = i
        elif col == "vote_average":
            idxRating = i
        elif col == "revenue":
            idxRev = i
        elif col == "budget":
            idxBudget = i
        elif col == "popularity":
            idxPop = i
        elif col == "release_date":
            idxRelease = i
        elif col == "genres":
            idxGenre = i
    return idxTitle, idxRating, idxRelease, idxRev, idxBudget, idxPop, idxGenre

def count_total_rows(dataset_path):
    total = 0
    with open(dataset_path, "r", encoding="latin-1") as f:
        reader = csv.reader(f)
        try:
            _ = next(reader)
        except StopIteration:
            return 0
        for _ in reader:
            total += 1
    return total

def mpi_gpu_movie_analysis(dataset_path, partition_mode="row", block_size_rows=1024):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if rank == 0:
        print(">>> [DEBUG] mpi_gpu_movie_analysis started", flush=True)
        print(f">>> [DEBUG] dataset_path = {dataset_path}", flush=True)
        print(f">>> [DEBUG] partition_mode = {partition_mode}, block_size_rows = {block_size_rows}", flush=True)

    total_start = MPI.Wtime()
    comm_time_local = 0.0
    comp_time_local = 0.0
    gpu_compute_only = 0.0

    if rank == 0:
        total_rows = count_total_rows(dataset_path)
        print(f">>> [DEBUG] Rank 0 counted total_rows = {total_rows}", flush=True)
    else:
        total_rows = None

    t0 = MPI.Wtime()
    if size > 1:
        if rank == 0:
            reqs = []
            for dest in range(1, size):
                req = comm.isend(total_rows, dest=dest, tag=11)
                reqs.append(req)
            for req in reqs:
                req.wait()
        else:
            req = comm.irecv(source=0, tag=11)
            total_rows = req.wait()
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    if total_rows == 0:
        if rank == 0:
            print("Empty file.", flush=True)
        return

    if partition_mode == "row":
        base = total_rows // size
        rem  = total_rows % size
        start_row = rank * base + min(rank, rem)
        end_row   = start_row + base + (1 if rank < rem else 0)
    elif partition_mode == "block-cyclic":
        start_row = 0
        end_row = total_rows
    else:
        raise ValueError("partition_mode must be 'row' or 'block-cyclic'")

    print(f">>> [DEBUG] Rank {rank} will process rows in [{start_row}, {end_row}) with mode={partition_mode}",
          flush=True)

    if rank == 0:
        print(f"[MPI GPU] Using dataset: {dataset_path}", flush=True)
        print(f"[MPI GPU] Total rows (movies) = {total_rows}", flush=True)
        print(f"[MPI GPU] Partition mode = {partition_mode}, block_size_rows = {block_size_rows}", flush=True)
        print(f"[MPI GPU] World size (processes) = {size}", flush=True)

    barrier_req = comm.Ibarrier()
    barrier_req.Wait()

    titles_local = []
    years_local = []
    ratings_local = []
    revenues_local = []
    pops_local = []
    global_index_local = []

    genre_to_id_local = {}
    id_to_genre_local = []
    genre_ids_local = []
    genre_ratings_local = []

    comp_start_cpu = MPI.Wtime()

    with open(dataset_path, "r", encoding="latin-1") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            header = []

        idxTitle, idxRating, idxRelease, idxRev, idxBudget, idxPop, idxGenre = get_column_indices(header)

        global_row_idx = 0

        for row in reader:
            if len(row) < len(header):
                continue

            if partition_mode == "row":
                if not (start_row <= global_row_idx < end_row):
                    global_row_idx += 1
                    continue
            else:
                block_id = global_row_idx // block_size_rows
                owner = block_id % size
                if owner != rank:
                    global_row_idx += 1
                    continue

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles_local.append(title)
            years_local.append(year)
            ratings_local.append(rating)
            revenues_local.append(rev)
            pops_local.append(pop)
            global_index_local.append(global_row_idx)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id_local and len(genre_to_id_local) < MAX_GENRES:
                        gid = len(id_to_genre_local)
                        genre_to_id_local[first] = gid
                        id_to_genre_local.append(first)
                    gid = genre_to_id_local.get(first, None)
                    if gid is not None:
                        genre_ids_local.append(gid)
                        genre_ratings_local.append(rating)

            global_row_idx += 1

    comp_end_cpu = MPI.Wtime()

    print(f">>> [DEBUG] Rank {rank} local rows read = {len(titles_local)}", flush=True)

    years_np = np.array(years_local, dtype=np.int32)
    ratings_np = np.array(ratings_local, dtype=np.float64)
    revenues_np = np.array(revenues_local, dtype=np.float64)
    pops_np = np.array(pops_local, dtype=np.float64)

    if len(id_to_genre_local) > 0:
        genre_ids_np = np.array(genre_ids_local, dtype=np.int32)
        genre_ratings_np = np.array(genre_ratings_local, dtype=np.float64)
    else:
        genre_ids_np = None
        genre_ratings_np = None

    if years_np.size > 0:
        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if genre_ids_np is not None:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(year_idx, weights=rating_valid, minlength=YEAR_RANGE).astype(cp.float64)
        d_count_year = cp.bincount(year_idx, minlength=YEAR_RANGE).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=len(id_to_genre_local)
            ).astype(cp.float64)
            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=len(id_to_genre_local)
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_end = time.perf_counter()
        gpu_compute_only = gpu_end - gpu_start

        sum_year_local = cp.asnumpy(d_sum_year)
        count_year_local = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_local = cp.asnumpy(d_sum_genre)
            count_genre_local = cp.asnumpy(d_count_genre)
        else:
            sum_genre_local = np.zeros(0, dtype=np.float64)
            count_genre_local = np.zeros(0, dtype=np.int64)

        local_top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        local_top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)
    else:
        sum_year_local = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_local = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_local = np.zeros(0, dtype=np.float64)
        count_genre_local = np.zeros(0, dtype=np.int64)
        local_top_rev_idx = np.array([], dtype=int)
        local_top_pop_idx = np.array([], dtype=int)

    comp_time_local = (comp_end_cpu - comp_start_cpu) + gpu_compute_only

    local_top_rev = []
    for idx in local_top_rev_idx:
        if 0 <= idx < len(titles_local):
            local_top_rev.append((
                revenues_local[idx],
                titles_local[idx],
                global_index_local[idx]
            ))

    local_top_pop = []
    for idx in local_top_pop_idx:
        if 0 <= idx < len(titles_local):
            local_top_pop.append((
                pops_local[idx],
                titles_local[idx],
                years_local[idx],
                global_index_local[idx]
            ))

    sum_year_global = np.zeros_like(sum_year_local)
    count_year_global = np.zeros_like(count_year_local)

    t0 = MPI.Wtime()
    comm.Reduce(sum_year_local, sum_year_global, op=MPI.SUM, root=0)
    comm.Reduce(count_year_local, count_year_global, op=MPI.SUM, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    t0 = MPI.Wtime()
    all_top_rev = comm.gather(local_top_rev, root=0)
    all_top_pop = comm.gather(local_top_pop, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    local_genre_payload = (id_to_genre_local, sum_genre_local, count_genre_local)
    t0 = MPI.Wtime()
    all_genre_payloads = comm.gather(local_genre_payload, root=0)
    t1 = MPI.Wtime()
    comm_time_local += (t1 - t0)

    total_end = MPI.Wtime()
    total_time_local = total_end - total_start

    comm_time_global = comm.reduce(comm_time_local, op=MPI.MAX, root=0)
    comp_time_global = comm.reduce(comp_time_local, op=MPI.MAX, root=0)
    gpu_time_global  = comm.reduce(gpu_compute_only, op=MPI.MAX, root=0)
    total_time_global = comm.reduce(total_time_local, op=MPI.MAX, root=0)

    if rank == 0:
        flat_top_rev = []
        for lst in all_top_rev:
            flat_top_rev.extend(lst)
        flat_top_rev.sort(key=lambda x: x[0], reverse=True)
        flat_top_rev = flat_top_rev[:MAX_TOP]

        flat_top_pop = []
        for lst in all_top_pop:
            flat_top_pop.extend(lst)
        flat_top_pop.sort(key=lambda x: x[0], reverse=True)
        flat_top_pop = flat_top_pop[:MAX_TOP]

        global_genre_map = {}
        global_sum_genre = []
        global_count_genre = []

        for (names, sums, counts) in all_genre_payloads:
            for i, gname in enumerate(names):
                s = float(sums[i])
                c = int(counts[i])
                if gname not in global_genre_map:
                    gid = len(global_genre_map)
                    global_genre_map[gname] = gid
                    global_sum_genre.append(s)
                    global_count_genre.append(c)
                else:
                    gid = global_genre_map[gname]
                    global_sum_genre[gid] += s
                    global_count_genre[gid] += c

        print("=====================================", flush=True)
        print("GPU Version – TMDB Movie Analysis (MPI + GPU, Non-blocking bcast+barrier)", flush=True)
        print(f"Movies processed: {total_rows}\n", flush=True)

        print(f"Top {MAX_TOP} Movies by Revenue:", flush=True)
        for i, (rev, title, gidx) in enumerate(flat_top_rev, start=1):
            print(f"{i:2d}. \"{title}\" | Revenue: {rev:.2f}", flush=True)

        print(f"\nTop {MAX_TOP} Movies by Popularity:", flush=True)
        for i, (pop, title, year, gidx) in enumerate(flat_top_pop, start=1):
            print(f"{i:2d}. \"{title}\" | Year:{year} | Popularity: {pop:.2f}", flush=True)

        print("\nAverage Rating per Year:", flush=True)
        print("Year  Count  Avg", flush=True)
        for y in range(FIRST_YEAR, LAST_YEAR + 1):
            idx = y - FIRST_YEAR
            cnt = int(count_year_global[idx])
            if cnt > 0:
                avg = sum_year_global[idx] / cnt
                print(f"{y:4d}  {cnt:6d}  {avg:.3f}", flush=True)

        print("\nAverage Rating per Genre:", flush=True)
        print("Genre                 Count    Avg", flush=True)
        for gname, gid in global_genre_map.items():
            cnt = int(global_count_genre[gid])
            if cnt > 0:
                avg = global_sum_genre[gid] / cnt
                print(f"{gname:<20s} {cnt:6d}   {avg:.3f}", flush=True)

        print("\n--- Timing Breakdown (MPI + GPU, Non-blocking bcast+barrier) ---", flush=True)
        print(f"Computation Time (max over ranks, CPU+GPU): {comp_time_global:.6f} s", flush=True)
        print(f"Communication Time (max over ranks)       : {comm_time_global:.6f} s", flush=True)
        print(f"GPU Compute Only (max over ranks)         : {gpu_time_global:.6f} s", flush=True)
        print(f"Total Execution Time (max over ranks)     : {total_time_global:.6f} s", flush=True)
        print("=====================================", flush=True)

if __name__ == "__main__":
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()

    if rank == 0:
        print(">>> [DEBUG] __main__ reached", flush=True)
        print(f">>> [DEBUG] sys.argv = {sys.argv}", flush=True)

    if len(sys.argv) < 2:
        if rank == 0:
            print("Usage:", flush=True)
            print("  mpirun -np <P> python mpi_gpu_tmdb.py <dataset_path> [row|block-cyclic] [block_size_rows]", flush=True)
        sys.exit(0)

    dataset_path = sys.argv[1]
    if len(sys.argv) >= 3:
        partition_mode = sys.argv[2]
    else:
        partition_mode = "row"

    if len(sys.argv) >= 4:
        block_size_rows = int(sys.argv[3])
    else:
        block_size_rows = 1024

    mpi_gpu_movie_analysis(dataset_path,
                           partition_mode=partition_mode,
                           block_size_rows=block_size_rows)


Overwriting mpinon_gpu_tmdb.py


In [ ]:
!python3 -u mpinon_gpu_tmdb.py /content/TMDB_movie_dataset_v112.csv row

>>> [DEBUG] __main__ reached
>>> [DEBUG] sys.argv = ['mpinon_gpu_tmdb.py', '/content/TMDB_movie_dataset_v112.csv', 'row']
>>> [DEBUG] mpi_gpu_movie_analysis started
>>> [DEBUG] dataset_path = /content/TMDB_movie_dataset_v112.csv
>>> [DEBUG] partition_mode = row, block_size_rows = 1024
>>> [DEBUG] Rank 0 counted total_rows = 1048575
>>> [DEBUG] Rank 0 will process rows in [0, 1048575) with mode=row
[MPI GPU] Using dataset: /content/TMDB_movie_dataset_v112.csv
[MPI GPU] Total rows (movies) = 1048575
[MPI GPU] Partition mode = row, block_size_rows = 1024
[MPI GPU] World size (processes) = 1
>>> [DEBUG] Rank 0 local rows read = 1048575
GPU Version – TMDB Movie Analysis (MPI + GPU, Non-blocking bcast+barrier)
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 â*: .ï½¡. o(â§â½â¦)o .ï½¡.:*â UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue